In [ ]:
import numpy as np
import math
from scipy.constants import m_e, c, e, physical_constants
from scipy.special import gamma
import numba
import matplotlib.pyplot as plt

In [ ]:
Uion = 13.6*e  # ionization energy in Joules, for Hydrogen

# Calculate the ADK prefactors (See Chen, JCP 236 (2013), equation (2))
# - Scalars
alpha = physical_constants['fine-structure constant'][0]
r_e = physical_constants['classical electron radius'][0]
wa = alpha**3 * c / r_e
Ea = m_e*c**2/e * alpha**4/r_e
# - Arrays (one element per ionization level)
UH = 13.6*e
Z = 1
n_eff = Z * np.sqrt( UH/Uion )
l_eff = n_eff - 1
C2 = 2**(2*n_eff) / (n_eff * gamma(n_eff+l_eff+1) * gamma(n_eff-l_eff))
# For now, we assume l=0, m=0
adk_power = - (2*n_eff - 1)
adk_prefactor = wa * C2 * ( Uion/(2*UH) ) \
    * ( 2*(Uion/UH)**(3./2)*Ea )**(2*n_eff - 1)
adk_exp_prefactor = -2./3 * ( Uion/UH )**(3./2) * Ea

In [ ]:
@numba.njit
def get_fraction_and_temperature( a0, tau, lambd, ell, npts_per_wavelength=80 ):

    # Precompute a few things
    dt = lambd/c/npts_per_wavelength
    omega = 2*np.pi*c/lambd
    E0 = m_e*omega*c/e
    inv_tau2 = 1./tau**2
    # Check the ellipticity
    assert len(ell) == 2
    assert abs(ell[0]**2 + ell[1]**2 - 1) < 1.e-10 # Check that the ellipticity is normalized

    t = -3*tau # Start at 3 sigma
    ioniz_frac = 0
    kin_energy = 0

    while t < 3*tau:

        # Get the electric field and vector potential
        a_env = a0 * math.exp(-inv_tau2*t**2)
        a = a_env * math.sqrt( ell[0]**2*np.cos(omega*t)**2 + ell[1]**2*np.sin(omega*t)**2 )
        E = E0 * a_env * math.sqrt( ell[0]**2*np.sin(omega*t)**2 + ell[1]**2*np.cos(omega*t)**2 )                           

        # Get the ionization rate
        w = 0
        if E > 0:
            w = adk_prefactor * E**adk_power * math.exp( adk_exp_prefactor/E )
        dp = 1 - math.exp( -w*dt )

        # increment the ionization fraction
        kin_energy += (1 - ioniz_frac)*dp * m_e*c**2/e * (math.sqrt( 1 + a**2 ) - 1)
        ioniz_frac += (1 - ioniz_frac)*dp
        
        t += dt

    T = kin_energy/(3/2*ioniz_frac)

    return ioniz_frac, T


In [ ]:
a0 = 0.05
tau = 30e-15
lambd = 0.8e-6
dt = lambd/c/80
ell = np.array([0,1]) 

get_fraction_and_temperature( a0, tau, lambd, ell )

In [ ]:
tau_arr = 10**np.linspace(-14, -11.5, 40)

a0 = 0.05
ell = np.array([0,1]) 
T_arr = [ get_fraction_and_temperature( a0, tau, lambd, ell )[1] for tau in tau_arr ]
plt.loglog( tau_arr, T_arr, 'b-', label='Linear polarization, a0=0.05' )

a0 = 0.025
ell = np.array([0,1]) 
T_arr = [ get_fraction_and_temperature( a0, tau, lambd, ell )[1] for tau in tau_arr ]
plt.loglog( tau_arr, T_arr, 'b--', label='Linear polarization, a0=0.025' )

a0 = 0.05
ell = np.array([1,1])/2**.5 
T_arr = [ get_fraction_and_temperature( a0, tau, lambd, ell )[1] for tau in tau_arr ]
plt.loglog( tau_arr, T_arr, 'r-', label='Circular polarization, a0=0.05' )

a0 = 0.025
ell = np.array([1,1])/2**.5 
T_arr = [ get_fraction_and_temperature( a0, tau, lambd, ell )[1] for tau in tau_arr ]
plt.loglog( tau_arr, T_arr, 'r--', label='Circular polarization, a0=0.025' )

plt.grid()
plt.legend(loc=0)

plt.xlabel('Laser duration (s)')
plt.ylabel('Temperature (eV)')